# Week 6 · Friday: Perceptrons and TensorFlow Basics

On Wednesday a neuron ran with weights we made up, and yesterday we met tensors and graphs in TensorFlow. Today one neuron learns its own weights:

1. **Activation functions:** the function at the end of every neuron
2. **The perceptron:** a weighted sum, a bias and a step, on the AND table
3. **Training a perceptron:** the learning rule, epoch by epoch
4. **TensorFlow basics:** constants, variables, and placeholders as history
5. **Creating a model:** the same neuron in Keras
6. **Graph visualization:** a model's structure as a table and as a picture

Run the next cell first. It holds the imports and plotting helpers we reuse all day, and it fixes the random seeds so your numbers match the ones on the screen.

In [ ]:
# ---------- Setup and plotting helpers (run once; no need to read live) ----------
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import tensorflow as tf

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.keras.utils.set_random_seed(RANDOM_STATE)   # seeds Python, NumPy and TensorFlow together


def plot_points(X, labels=None, title="", xlabel="Feature 1", ylabel="Feature 2",
                names=None, centers=None, ax=None):
    """Scatter 2D points, coloured by label if given. Label -1 is drawn as noise (DBSCAN)."""
    show = ax is None
    if ax is None:
        _, ax = plt.subplots(figsize=(8, 6))
    if labels is None:
        ax.scatter(X[:, 0], X[:, 1], s=45, edgecolor="black", alpha=0.85)
    else:
        for value in np.unique(labels):
            mask = labels == value
            if value == -1:
                ax.scatter(X[mask, 0], X[mask, 1], s=70, marker="x", color="black", label="Noise (-1)")
            else:
                label = names[value] if names is not None else f"{value}"
                ax.scatter(X[mask, 0], X[mask, 1], s=45, edgecolor="black", alpha=0.85, label=label)
        ax.legend()
    if centers is not None:
        ax.scatter(centers[:, 0], centers[:, 1], s=300, marker="X", color="red",
                   edgecolor="black", linewidth=2, label="Centroids")
        ax.legend()
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.3)
    if show:
        plt.show()


def plot_decision_boundary(model, X, y, title="", xlabel="Feature 1", ylabel="Feature 2",
                           names=None, ax=None):
    """Colour the background by what the model predicts at every point of a fine grid."""
    show = ax is None
    if ax is None:
        _, ax = plt.subplots(figsize=(8, 6))
    xx, yy = np.meshgrid(np.linspace(X[:, 0].min() - 1, X[:, 0].max() + 1, 300),
                         np.linspace(X[:, 1].min() - 1, X[:, 1].max() + 1, 300))
    grid_predictions = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    classes = np.unique(y)
    colours = ListedColormap(plt.cm.tab10.colors[:len(classes)])   # same colours as the points
    ax.contourf(xx, yy, grid_predictions, levels=np.arange(len(classes) + 1) - 0.5, alpha=0.25, cmap=colours)
    plot_points(X, y, title, xlabel, ylabel, names=names, ax=ax)
    if show:
        plt.show()


def plot_line_chart(xs, ys, title, xlabel, ylabel):
    """A simple line chart with markers, for the elbow and silhouette plots."""
    plt.figure(figsize=(8, 5))
    plt.plot(xs, ys, marker="o", linewidth=2)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.xticks(xs)
    plt.grid(alpha=0.3)
    plt.show()


print(f"TensorFlow {tf.__version__}, Keras {tf.keras.__version__}")
print("Setup complete")

## 1. Activation functions

On Wednesday a neuron did two things. First it multiplied each input by its **weight** and added them up, plus a **bias** (a number that shifts the total up or down). Call that sum `z`:

```
z = w1*x1 + w2*x2 + ... + b
```

Then it passed `z` through an **activation function**: the function that turns the sum into the neuron's output. Its shape decides what the neuron can say: yes or no, a probability, or "how much".

Three common ones live in `tf.keras.activations`. We compute each one with Keras and with its numpy formula, to check they agree.

In [ ]:
z = tf.constant([-2.0, 0.0, 2.0])
z_np = z.numpy()

sigmoid = tf.keras.activations.sigmoid(z).numpy()
tanh = tf.keras.activations.tanh(z).numpy()
relu = tf.keras.activations.relu(z).numpy()

print(f"z        {z_np}")
print(f"sigmoid  {sigmoid.round(3)}  same as 1 / (1 + e^-z)? {np.allclose(sigmoid, 1 / (1 + np.exp(-z_np)))}")
print(f"tanh     {tanh.round(3)}  same as np.tanh(z)?     {np.allclose(tanh, np.tanh(z_np))}")
print(f"relu     {relu.round(3)}  same as max(0, z)?      {np.allclose(relu, np.maximum(0, z_np))}")

- **Sigmoid** squeezes any `z` into the range 0 to 1, so its output reads as a probability. It is the usual output for a yes-or-no label. At `z = 0` it gives 0.5.
- **Tanh** squeezes `z` into the range -1 to 1, centred on 0.
- **ReLU** (rectified linear unit) keeps positive values and turns every negative value into 0: `max(0, z)`. It is cheap to compute, which is why the hidden layers of week 7's networks use it.

The perceptron in section 2 uses a fourth one, the **step function**: it outputs 1 or 0 and nothing in between. `tf.keras.activations` has no step, so we write it in numpy.

> **Our rule for the whole day: at `z = 0` the step outputs 1.** The perceptron cells and the paper exercise after the break use the same rule.

In [ ]:
def step(z):
    return np.where(z >= 0, 1, 0)      # 1 at z = 0: our rule for the day

print(step(np.array([-0.5, 0.0, 0.5])))

Now all four as curves. The thick line is Keras, the dashed line is the numpy formula drawn on top of it.

In [ ]:
z = np.linspace(-5, 5, 201).astype("float32")      # Try changing it: np.linspace(-20, 20, 801)

curves = [
    ("sigmoid", tf.keras.activations.sigmoid(z).numpy(), 1 / (1 + np.exp(-z))),
    ("tanh", tf.keras.activations.tanh(z).numpy(), np.tanh(z)),
    ("ReLU", tf.keras.activations.relu(z).numpy(), np.maximum(0, z)),
    ("step (numpy only)", None, step(z)),
]

fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
for ax, (name, keras_values, numpy_values) in zip(axes, curves):
    if keras_values is not None:
        ax.plot(z, keras_values, linewidth=6, alpha=0.4, label="tf.keras.activations")
    ax.plot(z, numpy_values, "k--", linewidth=1.5, label="numpy formula")
    ax.axhline(0, color="grey", linewidth=0.8)
    ax.axvline(0, color="grey", linewidth=0.8)
    ax.set_title(name)
    ax.set_xlabel("z")
    ax.grid(alpha=0.3)
    ax.legend()
plt.tight_layout()
plt.show()

- The Keras curve and the numpy curve lie on top of each other: the library computes the formula, nothing more.
- Sigmoid and tanh **flatten out** at both ends. Out there, a big change in `z` barely moves the output. Try changing the range to -20 to 20 in the cell above and see how much of each curve is flat. ReLU stays at 0 on the left and keeps climbing on the right.
- The step jumps from 0 to 1 at `z = 0` and is flat everywhere else. Keep that in mind for section 5.

For more than two classes, the output layer uses **softmax**: it turns a list of scores, one per class, into probabilities that add up to 1. It is the many-class version of a single sigmoid output.

In [ ]:
scores = tf.constant([[1.0, 2.0, 3.0]])               # one score per class
probabilities = tf.keras.activations.softmax(scores)

print(f"Probabilities: {probabilities.numpy().round(3)}")
print(f"Sum:           {float(tf.reduce_sum(probabilities)):.3f}")

**Why the function has to be non-linear.** A **linear** function only multiplies and adds, so its graph is a straight line (or a flat plane). A weighted sum of weighted sums is still one weighted sum. Stack two layers with no activation and they collapse into a single layer. Here are two layers as two weight matrices, multiplied rows times columns as on Wednesday:

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
W1 = rng.normal(size=(3, 2))        # layer 1: 2 inputs -> 3 outputs
W2 = rng.normal(size=(1, 3))        # layer 2: 3 inputs -> 1 output
x = np.array([1.0, 2.0])

two_layers = W2 @ (W1 @ x)          # layer 1, then layer 2, no activation
one_layer = (W2 @ W1) @ x           # one combined weight matrix

print(f"Two layers, no activation: {two_layers.round(4)}")
print(f"One combined layer:        {one_layer.round(4)}")
print(f"Combined weights, shape {(W2 @ W1).shape}: {(W2 @ W1).round(3)}")

Same answer: without an activation, the second layer added nothing. The bend in sigmoid, tanh or ReLU is what lets stacked layers learn shapes a single straight line cannot. Week 7 builds those stacks; today we stay with one neuron.

## 2. The perceptron

A **perceptron** is the simplest neuron: a weighted sum of its inputs, plus a bias, passed through the step. Its output is 1 or 0.

### 2.1 Weighted sum, bias and step

Our example is the **AND** table: two inputs that are each 0 or 1, and a label that is 1 only when both inputs are 1. With weights we pick by hand, `w = [1, 1]` and `b = -1.5`, the sum is `z = x1 + x2 - 1.5`, which reaches 0 only on the last row.

In [ ]:
X_and = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_and = np.array([0, 0, 0, 1])          # AND: 1 only when both inputs are 1

w = np.array([1.0, 1.0])
b = -1.5                                # Try changing it: -0.5

z = X_and @ w + b                       # one weighted sum per row
for row, z_row, output, label in zip(X_and, z, step(z), y_and):
    print(f"x={row}  z={z_row:+.1f}  output={output}  label={label}")

fig, ax = plt.subplots(figsize=(6, 5))
plot_points(X_and, y_and, "AND, and the line where z = 0", "x1", "x2", ax=ax)
x1 = np.linspace(-0.5, 1.5, 50)
ax.plot(x1, -(w[0] * x1 + b) / w[1], "k--", label="z = 0")   # solve w1*x1 + w2*x2 + b = 0 for x2
ax.set_xlim(-0.5, 1.5)
ax.set_ylim(-0.5, 1.5)
ax.legend()
plt.show()

- The perceptron's **decision boundary** is the set of points where `z = 0`. With two inputs it is a straight line: points on one side get 1, points on the other side get 0.
- The **weights** tilt the line and the **bias** shifts it.
- Try changing the bias to -0.5. The outputs become 0, 1, 1, 1, the line slides down, and the same neuron now computes OR. Put it back to -1.5.

### 2.2 Parameters, and training on the AND table

The weights and the bias are the model's **parameters**: training learns them from the data. Two other numbers are chosen by you before training starts. They are **hyperparameters**:

- **Learning rate:** the size of each correction.
- **Epochs:** how many times training goes through the data. One **epoch** is one full pass over every row.

The **perceptron learning rule** goes row by row. If the output is right, nothing changes. If it is wrong:

```
error = label - output              # +1 if it said 0 but should say 1, -1 the other way round
w     = w + learning_rate * error * x
b     = b + learning_rate * error
```

The cell below starts from `w = [0, 0]`, `b = 0` and learning rate 1. It takes the rows in table order, uses the step rule from section 1 (output 1 at `z = 0`), and prints the weights and bias after each epoch, stopping after the first epoch with no mistakes.

**Work out epoch 1 on paper first**, then run the cell and check your answer against its first line.

In [ ]:
w = np.zeros(2)               # starting weights [0, 0]
b = 0.0                       # starting bias
learning_rate = 1.0           # Try changing it: 0.5

for epoch in range(1, 11):
    errors = 0
    for x, label in zip(X_and, y_and):          # the rows in table order
        z = x @ w + b
        output = 1 if z >= 0 else 0               # the step: 1 at z = 0
        if output != label:
            error = label - output
            w = w + learning_rate * error * x
            b = b + learning_rate * error
            errors += 1
    print(f"epoch {epoch}: w={w.tolist()} b={float(b)} errors={errors}")
    if errors == 0:
        break

- Epoch 1 makes two mistakes. Row (0, 0) has `z = 0`, so the step says 1 and the label is 0. Row (1, 1) has `z = -1`, so the step says 0 and the label is 1.
- On row (0, 0) only the bias moves. Each weight moves by `learning_rate * error * input`, and both inputs are 0. The bias is what lets the line move when every input is 0.
- Training stops in epoch 6 at `w = [2.0, 1.0]`, `b = -3.0`. That is not our hand-picked `[1, 1]` and `-1.5`, and it is still correct: any line that separates the two classes is a right answer.
- Try changing the learning rate to 0.5 and rerun. Every printed number halves, and training still stops in epoch 6. From zero weights, every update adds `learning_rate` times the same `±x`, so the learning rate only rescales `w` and `b`: the predictions and the line stay the same. Put it back to 1.0.

## 3. Training a perceptron

The AND table has four rows. Now the same rule on 100 points from `make_blobs`: two groups of points, labelled 0 and 1.

### 3.1 The learning rule on two blobs

One change from 2.2: the weights start **small and random**, from a fixed seed, instead of at zero. From zero, the learning rate only rescales `w` and `b` (the AND cell above shows it), so every prediction and the drawn boundary would come out the same at any learning rate, and changing it would show nothing.

The cell redraws the boundary after each epoch.

In [ ]:
from sklearn.datasets import make_blobs

X_blobs, y_blobs = make_blobs(n_samples=100, centers=2, cluster_std=1.0, random_state=RANDOM_STATE)

learning_rate = 1.0                      # Try changing it: 0.01
epochs = 5
rng = np.random.default_rng(RANDOM_STATE)
w = rng.normal(0, 0.1, 2)                # small random start, the same on every run
b = 0.0

x1 = np.linspace(X_blobs[:, 0].min() - 1, X_blobs[:, 0].max() + 1, 50)
fig, axes = plt.subplots(1, epochs, figsize=(4.5 * epochs, 4.5))
mistakes_per_epoch = []
for epoch, ax in enumerate(axes, start=1):
    mistakes = 0
    for x, label in zip(X_blobs, y_blobs):
        output = 1 if x @ w + b >= 0 else 0           # the step: 1 at z = 0
        if output != label:
            w = w + learning_rate * (label - output) * x
            b = b + learning_rate * (label - output)
            mistakes += 1
    mistakes_per_epoch.append(mistakes)
    plot_points(X_blobs, y_blobs, f"After epoch {epoch}: {mistakes} mistakes", ax=ax)
    ax.plot(x1, -(w[0] * x1 + b) / w[1], "k--", linewidth=2)      # the line where z = 0
    ax.set_ylim(X_blobs[:, 1].min() - 1, X_blobs[:, 1].max() + 1)
plt.tight_layout()
plt.show()

print(f"Mistakes per epoch: {mistakes_per_epoch}")
print(f"Final weights {w.round(3)}, bias {float(b):.3f}")

In [ ]:
plot_line_chart(list(range(1, epochs + 1)), mistakes_per_epoch, "Mistakes per epoch", "Epoch", "Mistakes")

- The blobs are far apart, so the line lands between them during epoch 1 and never moves again: every later epoch makes no mistakes, and the rule changes nothing when the answer is right.
- Try changing the learning rate to 0.01, from the same seeded start. Epoch 1 makes one mistake instead of two and ends on a different line: with small corrections the random start has more say. Both runs end with the blobs separated. Put it back to 1.0.
- Then raise `cluster_std` on the `make_blobs` line, step by step, until the blobs overlap (5.0 does it). The line jumps every epoch and the mistakes never reach 0. That is not a bug: the rule only settles when one straight line can separate the classes, and no straight line separates overlapping points. Week 7 starts from that limit.
- Put `cluster_std` back to 1.0 and rerun the cell before 3.2, which reads these blobs.

### 3.2 scikit-learn's Perceptron agrees

scikit-learn's `Perceptron` runs the same rule: one correction after each row, only on mistakes. Its learning rate is `eta0` and its limit on epochs is `max_iter`.

In [ ]:
from sklearn.linear_model import Perceptron

clf = Perceptron(random_state=RANDOM_STATE)          # Try changing it: Perceptron(eta0=0.01, random_state=RANDOM_STATE)
clf.fit(X_blobs, y_blobs)

ours = np.where(X_blobs @ w + b >= 0, 1, 0)          # our loop's predictions from 3.1
print(f"scikit-learn accuracy on the blobs:        {clf.score(X_blobs, y_blobs)}")
print(f"Share of points where it agrees with ours: {np.mean(clf.predict(X_blobs) == ours)}")
print(f"scikit-learn: w={clf.coef_[0].round(3)} b={clf.intercept_[0]:.3f}")
print(f"our loop:     w={w.round(3)} b={float(b):.3f}")

plot_decision_boundary(clf, X_blobs, y_blobs, "scikit-learn Perceptron")

- Both numbers are 1.0: every point is classified correctly, and the two models put every point on the same side.
- The weights differ. Many lines separate these blobs, and scikit-learn visits the rows in a shuffled order each epoch, so it stops on a different one. What matches is the side each point lands on.
- Try adding `eta0=0.01`. Both numbers stay at 1.0: scikit-learn starts from zero weights, and from zero the learning rate only rescales the weights. That is why its documentation says the perceptron does not need a learning rate. Take `eta0` out again.

## 4. TensorFlow basics

Thursday's tensors were all made with `tf.constant`. A **tensor** is TensorFlow's array: it has a **shape** (its size along each axis) and a **dtype** (the type of its values), and each operation runs as soon as you run the line (**eager execution**). The perceptron loop kept its weights in a numpy array and replaced them after every mistake. TensorFlow keeps values that change in a different kind of object.

### 4.1 `tf.constant` against `tf.Variable`

- `tf.constant` makes an **immutable** tensor: once made, its values never change. The data you feed a model fits that, because it is read and never updated.
- `tf.Variable` holds values that change. `assign` replaces them, `assign_add` adds to them (`assign_sub` subtracts), and the shape and dtype stay fixed.

In [ ]:
c = tf.constant([1.0, 2.0])
v = tf.Variable([1.0, 2.0])
print(f"constant: shape {c.shape}, dtype {c.dtype.name}")

v.assign([5.0, 6.0])           # replace the values
v.assign_add([1.0, 1.0])       # add to them
print(f"variable after assign and assign_add: {v.numpy()}")

try:
    c.assign([5.0, 6.0])       # a constant has no assign
except AttributeError as err:
    print(f"constant: {type(err).__name__}, still {c.numpy()}")

A variable is **trainable** unless you say otherwise: training updates it. Weights and biases are trainable. A value that is state but must not be touched by training, such as a counter of training steps, is made with `trainable=False`.

In [ ]:
weights = tf.Variable([0.0, 0.0])
step_count = tf.Variable(0, trainable=False)      # state, not a weight
print(f"weights trainable: {weights.trainable}, step_count trainable: {step_count.trainable}")

new_values = [1.0, 2.0]                           # Try changing it: [1.0, 2.0, 3.0]
try:
    weights.assign(new_values)
    step_count.assign_add(1)
    print(f"weights {weights.numpy()}, steps {step_count.numpy()}")
except ValueError:
    print("ValueError: a variable keeps its shape")
print(f"weights: shape {weights.shape}, dtype {weights.dtype.name}")

- Training changes only the trainable variables.
- Try the three-number change: a `ValueError`, and the variable still has shape (2,). You can replace a weight vector's numbers, but you cannot turn two weights into three. Put it back.
- The perceptron's update, written in TensorFlow, would be `w.assign_add(learning_rate * error * x)`.

### 4.2 Placeholders as TensorFlow 1 history

The curriculum lists "constants, placeholders, variables". Placeholders belong to **TensorFlow 1**, which built a graph first and ran it later in a **session**. A **placeholder** was an empty slot in that graph for the input, and `feed_dict` filled the slot when the session ran. You will see this in old tutorials. It does not run in TensorFlow 2, so it stays in a text cell:

**TensorFlow 1 (history, do not run)**

```python
x = tf.placeholder(tf.float32, shape=[None])
doubled = x * 2.0
with tf.Session() as sess:
    print(sess.run(doubled, feed_dict={x: [1.0, 2.0]}))
```

In TensorFlow 2 the slot is a plain function argument, and `tf.function` (Thursday) turns the function into a graph when you want one:

In [ ]:
@tf.function
def double(x):
    return x * 2.0

print(double(tf.constant([1.0, 2.0])).numpy())      # Try changing it: tf.constant([10.0, 20.0, 30.0])

- No session and no `feed_dict`: the value is simply the argument.
- Try the three-number tensor: a new shape traces a new graph, as on Thursday, and nothing else changes.
- If an old tutorial shows `tf.placeholder`, `tf.Session`, `feed_dict` or `tf.global_variables_initializer()`, read it for the idea, then write the TensorFlow 2 version: function arguments, eager execution, `tf.function` and `tf.Variable`. The tensorflow.org guide "Migrate to TensorFlow 2" lists the replacement for each.

## 5. Creating a model

Now the same idea, one neuron, as a model built with **Keras**, the high-level API that ships with TensorFlow as `tf.keras`.

The data is scikit-learn's breast cancer set: 30 measurements per tumour and a yes-or-no label. We split it with Monday's `train_test_split` and scale it with `StandardScaler` (Tuesday's `### 5.2`), fitted on the training rows only, so nothing about the test rows leaks into training. Unscaled features make a single neuron train slowly.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

cancer = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(
    cancer.data, cancer.target, test_size=0.2, random_state=RANDOM_STATE, stratify=cancer.target
)
scaler = StandardScaler().fit(X_train)       # mean and spread from the training rows only
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

print(f"Labels: 0 = {cancer.target_names[0]}, 1 = {cancer.target_names[1]}")
print(f"Training rows {X_train.shape}, test rows {X_test.shape}")

- **`tf.keras.Sequential`** is a stack of layers that run in order.
- `tf.keras.Input(shape=(30,))` declares 30 inputs per row, so the weights exist straight away.
- **`Dense(1, activation="sigmoid")`** is one neuron. A **Dense layer** gives every output a weight for every input, adds a bias, and applies the activation. This one is the perceptron with a sigmoid where the step was.

In [ ]:
tf.keras.utils.set_random_seed(RANDOM_STATE)   # the same starting weights on every rerun

model = tf.keras.Sequential([
    tf.keras.Input(shape=(30,)),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])
print(f"Numbers this model learns: {model.count_params()}")

31 numbers: one weight per feature, plus one bias.

`compile` sets how the model will train:

- `optimizer="adam"`: **Adam** is the rule that nudges the weights a little after each **batch** (a small group of rows) to lower the loss. It is a refined version of the perceptron's correction; week 7 shows how the nudges are worked out.
- `loss="binary_crossentropy"`: the **loss** for a yes-or-no label, a number that grows as the predicted probability moves away from the true label. Training tries to make it small.
- `metrics=["accuracy"]`: also report the share of rows classified correctly.

`fit` then trains for a number of epochs and returns a **History** object holding the loss and metrics for each epoch.

In [ ]:
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

epochs = 50                                     # Try changing it: 5 (rerun the cell above first)
history = model.fit(X_train, y_train, epochs=epochs, verbose=0)
print(f"History keys: {sorted(history.history)}")
print(f"Training loss: epoch 1 {history.history['loss'][0]:.3f}, epoch {epochs} {history.history['loss'][-1]:.3f}")

plt.figure(figsize=(8, 5))
plt.plot(range(1, epochs + 1), history.history["loss"], marker="o", markersize=3, linewidth=2)
plt.title("Training loss per epoch")
plt.xlabel("Epoch")
plt.ylabel("Binary cross-entropy loss")
plt.grid(alpha=0.3)
plt.show()

The loss drops fast in the first epochs, then flattens out. That curve is measured on the training rows. The number to quote comes from `evaluate` on the test rows, which the model has never seen.

`predict` returns the sigmoid output for each row, a number between 0 and 1 that reads as the probability of class 1. We cut it at 0.5 to get a label.

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test loss:     {test_loss:.3f}")
print(f"Test accuracy: {test_accuracy:.3f}")

probabilities = model.predict(X_test[:5], verbose=0)
print(f"Probability of class 1:    {probabilities.ravel().round(3)}")
print(f"Predicted labels (>= 0.5): {(probabilities.ravel() >= 0.5).astype(int)}")
print(f"True labels:               {y_test[:5]}")

- `evaluate` returns the loss first, then the accuracy. Quote the test accuracy, not the training numbers and not the loss.
- A probability near 0 or 1 means the model is sure; one nearer 0.5 means it is less sure, even when the label comes out right.
- Try changing `epochs` to 5 in the `fit` cell. Rerun the cell that builds the model first, or `fit` carries on from the weights it already trained. The test loss comes out higher and the test accuracy no better: five passes of small nudges leave the weights near where they started. Put it back to 50 and rerun the build, `fit` and `evaluate` cells, because section 6 reads the trained model.

The neuron's 31 numbers live in its layer: the **kernel** (Keras's name for the layer's weight matrix, one weight per input) and the bias. In Keras 3 they are Keras `Variable`s. Keras 3 can run on more than one backend (the library that does the arithmetic), so it wraps its state in its own `Variable` type. On the TensorFlow backend, each one's `.value` is the `tf.Variable` underneath.

The path reads `sequential/dense/...` on a fresh run. If you rebuilt the model during the "Try changing", Keras numbers the new names (`sequential_1/dense_1/...` after one rebuild, `sequential_2/dense_2/...` after two); the weights are the same.

In [ ]:
layer = model.layers[0]
print(layer.bias)                                   # how a Keras Variable prints
print(f"kernel: path {layer.kernel.path}, shape {layer.kernel.shape}")
print(f"type of layer.kernel:       {type(layer.kernel).__name__}")
print(f"type of layer.kernel.value: {type(layer.kernel.value).__name__}")
print(f"layer.kernel.value is a tf.Variable: {isinstance(layer.kernel.value, tf.Variable)}")

kernel, bias = layer.get_weights()                  # the same numbers as numpy arrays
print(f"First five weights: {kernel[:5, 0].round(3)}, bias: {bias.round(3)}")

One sigmoid neuron trained on this loss is **logistic regression**, Monday's classifier. Let's fit Monday's `LogisticRegression` on the same scaled rows and compare.

In [ ]:
from sklearn.linear_model import LogisticRegression

logreg = LogisticRegression(max_iter=5000).fit(X_train, y_train)
keras_labels = (model.predict(X_test, verbose=0).ravel() >= 0.5).astype(int)

print(f"LogisticRegression: coef_ {logreg.coef_.shape}, intercept_ {logreg.intercept_.shape}"
      f" = {logreg.coef_.size + logreg.intercept_.size} numbers")
print(f"LogisticRegression test accuracy: {logreg.score(X_test, y_test):.3f}")
print(f"Keras neuron test accuracy:       {test_accuracy:.3f}")
print(f"Same label on {np.mean(keras_labels == logreg.predict(X_test)):.0%} of the test rows")

- The same 31 numbers: 30 coefficients and one intercept.
- The test accuracies are within a few points of each other, and the two models give the same label on most test rows. They differ a little because they were trained by different methods: scikit-learn's solver runs until the weights settle, while Adam stopped after 50 epochs of small nudges.
- Like the perceptron, one neuron can only draw one flat boundary through the 30 features. Week 7 stacks layers to get past that.

## 6. Graph visualization

A **computation graph** is a calculation drawn as nodes (layers or operations) joined by edges (the data passing between them). Keras gives two views of a model's structure: `model.summary()` prints a table, and `plot_model` draws a picture.

In [ ]:
model.summary()

- One row per layer. The Dense layer's output shape is `(None, 1)`: `None` is the number of rows, left open so the model takes any batch size.
- The Dense row shows 31 parameters, and "Trainable params" is 31: the model's weights and bias.
- "Total params" is bigger because, after training, it also counts "Optimizer params": numbers Adam keeps for its own bookkeeping. They are not part of the model's answer.

`tf.keras.utils.plot_model` draws the same structure as a diagram. It needs two extra tools, the `pydot` package and the `graphviz` program. Colab ships both, so there is nothing to install. On a laptop without them it raises an error asking for them, and that error says nothing about your model.

In [ ]:
tf.keras.utils.plot_model(model, show_shapes=True)      # Try changing it: show_shapes=False

- The input `(None, 30)` feeds the Dense layer, which outputs `(None, 1)`. Try `show_shapes=False`: the same boxes, without the shapes.
- This is the **layer structure**. Thursday's TensorBoard Graphs tab is a different, lower-level picture: the operations that actually ran. Notes step 7 traces this model into TensorBoard if you want to see both.

## Recap

| Idea | Type | Good at | Watch out for |
|---|---|---|---|
| Activation function | the function after the weighted sum | sigmoid: a probability for yes or no; ReLU: cheap, for hidden layers | sigmoid and tanh go flat for large `z`; the step has no slope; our step outputs 1 at `z = 0` |
| Perceptron | supervised, linear classifier | learning from its mistakes, one correction at a time | settles only when one straight line separates the classes |
| Learning rate, epochs | hyperparameters | setting the size of each correction and the number of passes | from zero weights the learning rate only rescales `w` and `b` |
| `tf.constant`, `tf.Variable` | TensorFlow values | data that never changes; weights that training updates | a variable keeps its shape and dtype; `trainable=False` for state |
| Placeholder and session | TensorFlow 1 history | reading old tutorials | not TensorFlow 2: use function arguments and `tf.function` |
| `Sequential` with `Dense(1, activation="sigmoid")` | supervised, Keras model | one neuron trained with `fit`, scored with `evaluate` | it is logistic regression; scale the features; rebuild before you refit |
| `model.summary()`, `plot_model` | model inspection | layers, shapes, parameter counts | `plot_model` needs pydot and graphviz; it is not the execution graph |

**Quick check**
1. What does an activation function do, and why does a network need a non-linear one?
2. What does our step function output at `z = 0`, and why must everyone training the same perceptron use the same rule?
3. Which of weights, bias, learning rate and epochs does training learn? Starting from zero weights, what does changing the learning rate change?
4. You raised `cluster_std` until the blobs overlapped. Why did the mistakes never reach 0?
5. What is the difference between `tf.constant` and `tf.Variable`, and what replaced TensorFlow 1's placeholders?
6. Why is one `Dense(1, activation="sigmoid")` neuron the same model as logistic regression, and how many numbers does it learn on the breast cancer data?